# 📹 Smart Camera AI - All-in-One Google Colab Notebook

Real-time Computer Vision system running on Google Colab Cloud GPU (NVIDIA T4):
- 🎯 **YOLO11 Object & Keypoint Detection**
- 💺 **Posture Classification** (Sitting vs. Standing with timestamps)
- ⏱️ **Dwell Timing & System Detection Coverage**
- 🌐 **Web Dashboard with Live Laptop Webcam Streaming**

---
### ⚡ Quick Setup Instructions:
1. **Runtime Type**: In Colab top menu, click `Runtime` ➔ `Change runtime type` ➔ select **T4 GPU**.
2. Run the cells below in order (1 ➔ 2 ➔ 3).

### Cell 1: ⚡ Check GPU & Install Dependencies
Run this cell once to verify GPU acceleration and install required libraries.

In [ ]:
# 1. Verify NVIDIA GPU
!nvidia-smi
import torch
print("--------------------------------------------------")
if torch.cuda.is_available():
    print(f"✅ GPU Ready: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: GPU is not active. Go to Runtime -> Change runtime type -> T4 GPU.")
print("--------------------------------------------------")

# 2. Install Python packages
!pip install -q ultralytics "uvicorn[standard]" fastapi pydantic sqlalchemy websockets python-multipart opencv-python-headless

# 3. Install Cloudflared for Linux
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("✅ Setup complete! Proceed to Cell 2.")

### Cell 2: 📥 Clone Fresh Code from GitHub
Downloads the latest Smart Camera code and verifies all components.

In [ ]:
# 1. Clean up any previous processes
!fuser -k 8000/tcp > /dev/null 2>&1
!pkill -9 -f uvicorn > /dev/null 2>&1
!pkill -9 -f cloudflared > /dev/null 2>&1

# 2. Clone repository
%cd /content
!rm -rf /content/SmartCamera
!git clone https://github.com/devaakashchauhan/SmartCamera.git
%cd /content/SmartCamera

# 3. Verify files and the webcam button
print("\n--- Verifying Frontend Controls ---")
!grep -n "Share Laptop Webcam" frontend/index.html
print("-----------------------------------")
print("✅ Code ready! Proceed to Cell 3.")

### Cell 3: 🚀 Launch Smart Camera Backend & Public Cloudflare Tunnel
Starts the FastAPI inference server on GPU, verifies health, and displays your live dashboard link.

In [ ]:
import subprocess
import time
import requests
import re

# 1. Ensure port 8000 is clean
!fuser -k 8000/tcp > /dev/null 2>&1
!pkill -9 -f uvicorn > /dev/null 2>&1
!pkill -9 -f cloudflared > /dev/null 2>&1
time.sleep(1)

# 2. Start FastAPI backend with Colab GPU in background
print("🚀 Starting Smart Camera backend server...")
server_log = open("/content/SmartCamera/server.log", "w")
server_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000", "--app-dir", "backend"],
    cwd="/content/SmartCamera",
    stdout=server_log,
    stderr=subprocess.STDOUT
)

# 3. Actively wait for backend to be online before starting tunnel
print("⏳ Waiting for AI backend to initialize on GPU...")
server_ready = False
for attempt in range(25):
    time.sleep(1)
    try:
        res = requests.get("http://127.0.0.1:8000/health", timeout=1)
        if res.status_code == 200:
            print(f"✅ Backend is online and healthy in {attempt + 1}s: {res.json()}")
            server_ready = True
            break
    except Exception:
        pass

if not server_ready:
    print("❌ Backend failed to start! Showing server.log traceback below:")
    print("=" * 60)
    try:
        with open("/content/SmartCamera/server.log") as f:
            print(f.read())
    except Exception:
        pass
    print("=" * 60)
    raise RuntimeError("Backend server did not start. Check server.log above.")

# 4. Launch Cloudflare Tunnel now that backend is 100% confirmed working
print("🌐 Opening Cloudflare public tunnel...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_t = time.time()

for line in tunnel_process.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    if time.time() - start_t > 30:
        break

if public_url:
    print("\n" + "=" * 65)
    print("🎉 YOUR SMART CAMERA DASHBOARD IS LIVE!")
    print(f"👉 Open Link: {public_url}")
    print("=" * 65)
    print("\nInstructions for live webcam:")
    print("  1. Click the link above on your laptop (or open in Incognito/Private tab).")
    print("  2. Click the green 'Share Laptop Webcam' button.")
    print("  3. Allow webcam permission.")
    print("  4. Watch real-time AI bounding boxes, sit/stand timers & dwell analytics!")
else:
    print("⚠️ Running tunnel directly in foreground:")
    !cloudflared tunnel --url http://127.0.0.1:8000

### Cell 4: 📋 View Live Server Logs (Optional)
Run this cell anytime to see live detections, posture events, and FPS output.

In [ ]:
!tail -n 40 /content/SmartCamera/server.log

### Cell 5: 🛑 Stop Server & Release Resources
Run this cell when you want to stop the system.

In [ ]:
!fuser -k 8000/tcp > /dev/null 2>&1
!pkill -9 -f uvicorn > /dev/null 2>&1
!pkill -9 -f cloudflared > /dev/null 2>&1
print("🛑 Smart Camera backend and Cloudflare tunnel stopped.")